In [13]:
class NhanVien:
    def __init__(self, ma_nv: str, luong_cb: float):
        self.__ma_nv = ma_nv        # private
        self.__luong_cb = luong_cb  # private

    # getter cho ma_nv
    def get_ma_nv(self):
        return self.__ma_nv

    # getter cho luong_cb
    def get_luong_cb(self):
        return self.__luong_cb

    def tinh_luong(self):
        return self.__luong_cb


class NhanVienTangCa(NhanVien):
    def tinh_luong(self):
        return self.get_luong_cb() * 1.2


class NhanVienChuyenChuyen(NhanVien):
    def tinh_luong(self):
        return self.get_luong_cb() + 500_000


if __name__ == "__main__":
    nv1 = NhanVien("NV01", 10_000_000)
    nv2 = NhanVienTangCa("NV02", 10_000_000)
    nv3 = NhanVienChuyenChuyen("NV03", 10_000_000)

    ds_nhan_vien = [nv1, nv2, nv3]

    for nv in ds_nhan_vien:
        print(f"Mã NV: {nv.get_ma_nv()} | Lương: {nv.tinh_luong():,.0f} VND")


Mã NV: NV01 | Lương: 10,000,000 VND
Mã NV: NV02 | Lương: 12,000,000 VND
Mã NV: NV03 | Lương: 10,500,000 VND


In [14]:
import numpy as np

san_luong = [12, None, 8, 15, None, 10]

# 1. Chuyển thành mảng NumPy, None -> np.nan, reshape 2x3
arr = np.array(san_luong, dtype=float)
arr = arr.reshape(2, 3)

print("Mảng sau reshape:")
print(arr)

# Tính trung bình bỏ qua NaN
mean_value = np.nanmean(arr)
print("Giá trị trung bình (bỏ NaN):", mean_value)

# Thay NaN bằng giá trị trung bình
arr_filled = np.where(np.isnan(arr), mean_value, arr)

print("Mảng sau khi thay NaN bằng trung bình:")
print(arr_filled)

# 2. Trích xuất phần tử > 9
filtered = arr_filled[arr_filled > 9]
print("Các phần tử > 9:", filtered)

# Giá trị nhỏ nhất trong tập vừa trích xuất
min_value = np.min(filtered)
print("Giá trị nhỏ nhất trong các phần tử > 9:", min_value)


Mảng sau reshape:
[[12. nan  8.]
 [15. nan 10.]]
Giá trị trung bình (bỏ NaN): 11.25
Mảng sau khi thay NaN bằng trung bình:
[[12.   11.25  8.  ]
 [15.   11.25 10.  ]]
Các phần tử > 9: [12.   11.25 15.   11.25 10.  ]
Giá trị nhỏ nhất trong các phần tử > 9: 10.0


CÂU 2

In [4]:
import pandas as pd
import sqlite3
import io

# Giả lập dữ liệu từ file nhan_su.csv bạn đã cung cấp
# (Khi chạy thực tế, bạn chỉ cần đảm bảo file nhan_su.csv nằm cùng thư mục)
csv_content = """Ma_NV,Phong_Ban,Thanh_Tich,Nam_Cong_Tac
NS001,KinhDoanh,85,2021
ns002,KinhDoanh,,2022
NS003,NhanSu,90,2023
NV004,NhanSu,70,2020
NS005,KyThuat,88,2021
ns006,KyThuat,,2024
NS007,KinhDoanh,95,2023
NV008,KyThuat,60,2019
NS009,NhanSu,,2022
NS010,KyThuat,92,2024
NS011,KinhDoanh,80,2021
nv012,NhanSu,75,2020"""

# Tạo file csv mẫu để code chạy được ngay
with open('nhan_su.csv', 'w', encoding='utf-8') as f:
    f.write(csv_content)

# 1. Đọc dữ liệu
df = pd.read_csv('nhan_su.csv')

# 2. Xử lý dữ liệu (Câu a, b)
# Lọc: Ma_NV chứa "NS" và Nam >= 2021
cond_ns = df['Ma_NV'].astype(str).str.contains('NS', case=False, na=False)
cond_nam = df['Nam_Cong_Tac'] >= 2021
df_filtered = df[cond_ns & cond_nam].copy()

# Điền giá trị thiếu (NaN) bằng trung bình cột Thanh_Tich
mean_val = df_filtered['Thanh_Tich'].mean()
df_filtered['Thanh_Tich'] = df_filtered['Thanh_Tich'].fillna(mean_val)

# 3. Tạo Pivot Table (Câu d)
pivot_table = pd.pivot_table(
    df_filtered,
    values='Thanh_Tich',
    index='Phong_Ban',
    columns='Nam_Cong_Tac',
    aggfunc='sum',
    fill_value=0
)

# 4. Xuất file kết quả

# Cách 1: Xuất ra SQLite (HeThong.db)
db_name = 'HeThong.db'
conn = sqlite3.connect(db_name)
pivot_table.to_sql('bang_luong_2024', conn, if_exists='replace')
conn.close()
print(f"✅ Đã xuất file cơ sở dữ liệu: {db_name}")

# Cách 2: Xuất ra CSV (để xem nhanh)
csv_output = 'bang_luong_2024.csv'
pivot_table.to_csv(csv_output)
print(f"✅ Đã xuất file CSV: {csv_output}")

# Hiển thị kết quả ra màn hình
print("\n--- KẾT QUẢ BẢNG PIVOT ---")
print(pivot_table)

✅ Đã xuất file cơ sở dữ liệu: HeThong.db
✅ Đã xuất file CSV: bang_luong_2024.csv

--- KẾT QUẢ BẢNG PIVOT ---
Nam_Cong_Tac   2021       2022  2023        2024
Phong_Ban                                       
KinhDoanh     165.0  88.333333  95.0    0.000000
KyThuat        88.0   0.000000   0.0  180.333333
NhanSu          0.0  88.333333  90.0    0.000000
